In [12]:
import pandas as pd
import spacy
from spacy.matcher import Matcher
from spacy import displacy
nlp = spacy.load('en_core_web_sm')

In [2]:
docs = ['What is your name?', 
        'Is coffee serving here?', 
        'Is she being promoted as new assistant?',
        'Women are said to live longer than men.']

#### Checking for 'nsubjpass' dependency

In [3]:
for doc in docs:
    print({doc: [tok.dep_ for tok in nlp(doc)]})

{'What is your name?': ['attr', 'ROOT', 'poss', 'nsubj', 'punct']}
{'Is coffee serving here?': ['ROOT', 'attr', 'advcl', 'advmod', 'punct']}
{'Is she being promoted as new assistant?': ['aux', 'nsubjpass', 'auxpass', 'ROOT', 'prep', 'amod', 'pobj', 'punct']}
{'Women are said to live longer than men.': ['nsubjpass', 'auxpass', 'ROOT', 'aux', 'xcomp', 'advmod', 'prep', 'pobj', 'punct']}


In [22]:
nsubjpass = [{'DEP':'nsubjpass'}]
matcher_nsubjpass = Matcher(vocab= nlp.vocab)
matcher_nsubjpass.add('Rule', [nsubjpass])

In [28]:
matched_nsubjpass = list(filter(lambda x: matcher_nsubjpass(nlp(x)), docs))
matched_nsubjpass

['Is she being promoted as new assistant?',
 'Women are said to live longer than men.']

In [29]:
mismatched_nsubjpass = list(filter(lambda x: not matcher_nsubjpass(nlp(x)), docs))
mismatched_nsubjpass

['What is your name?', 'Is coffee serving here?']

Visualizing dependencies

In [30]:
for doc in mismatched_nsubjpass:
    displacy.render(nlp(doc), style='dep')

In [35]:
for doc in matched_nsubjpass:
    displacy.render(nlp(doc), style='dep')

'What is your name?' is coming as an active voice sentence??? This should not have 'nsubjpass' dependency.

#### Checking for 'aux' dependency

In [21]:
docs_2 = [
    'Sofia is learning NLP.',
    'Eggs are laid by Hens.',
    'Mouse is eaten by a black cat.',
    'She has done her job productively.'
]

In [27]:
aux = [{'DEP':'aux'}]
matcher_aux = Matcher(vocab= nlp.vocab)
matcher_aux.add('Rule', [aux])

In [31]:
matched_aux = list(filter(lambda x: matcher_aux(nlp(x)), docs_2))
matched_aux

['Sofia is learning NLP.', 'She has done her job productively.']

In [32]:
mismatched_aux = list(filter(lambda x: not matcher_aux(nlp(x)), docs_2))
mismatched_aux

['Eggs are laid by Hens.', 'Mouse is eaten by a black cat.']

In [33]:
for doc in mismatched_aux:
    displacy.render(nlp(doc), style='dep')

In [34]:
for doc in matched_aux:
    displacy.render(nlp(doc), style='dep')

Dependency clarification:
 - 'aux': auxilliary verb in active voice
 - 'auxpass': auxilliary verb in passive voice

#### Dependency tag, POS tag checking for a given sentence

In [43]:
sent = 'JetAirways cancelled the flight this morning which was already late.'
pd.DataFrame([{'Token':tok.text, 'POS':tok.pos_, 'DEP':tok.dep_} for tok in nlp(sent)])

,Token,POS,DEP
0,JetAirways,PROPN,nsubj
1,cancelled,VERB,ROOT
2,the,DET,det
3,flight,NOUN,dobj
4,this,DET,det
5,morning,NOUN,npadvmod
6,which,PRON,nsubj
7,was,AUX,relcl
8,already,ADV,advmod
9,late,ADJ,acomp


#### Checking for children nodes in a given sentence

In [48]:
sent = 'It was the best of times and it was the worst of times.'
pd.DataFrame([{'Token':tok.text, 
               'POS':tok.pos_, 
               'DEP':tok.dep_, 
               'Child': (list(tok.children) if len(list(tok.children))>0 else 0)} 
              for tok in nlp(sent)])

,Token,POS,DEP,Child
0,It,PRON,nsubj,0
1,was,AUX,ROOT,"[It, best, and, was]"
2,the,DET,det,0
3,best,ADJ,attr,"[the, of]"
4,of,ADP,prep,[times]
5,times,NOUN,pobj,0
6,and,CCONJ,cc,0
7,it,PRON,nsubj,0
8,was,AUX,conj,"[it, worst, .]"
9,the,DET,det,0


In [45]:
displacy.render(nlp(sent), style='dep')

In [49]:
sent = 'Dole was defeated by Clinton.'
pd.DataFrame([{'Token':tok.text, 
               'POS':tok.pos_, 
               'DEP':tok.dep_, 
               'Child': (list(tok.children) if len(list(tok.children))>0 else 0)} 
              for tok in nlp(sent)])

,Token,POS,DEP,Child
0,Dole,PROPN,nsubjpass,0
1,was,AUX,auxpass,0
2,defeated,VERB,ROOT,"[Dole, was, by, .]"
3,by,ADP,agent,[Clinton]
4,Clinton,PROPN,pobj,0
5,.,PUNCT,punct,0
